# 02. The virtual populationRun `make population` first. This notebook reads `results/` and does not recompute thecohort.The question this notebook exists to make visible: **does anything measurable at baselinetell you who is going to over-respond?**

In [ ]:
import jsonfrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom hcmtwin.viz import stylestyle.apply()RESULTS = Path("../results")labelled = pd.read_csv(RESULTS / "population_labelled.csv.gz")summary = json.loads((RESULTS / "population_summary.json").read_text())print(json.dumps(summary, indent=1))

## Trial eligibilityThe over-response rate is only comparable to a published rate on a cohort built with thesame enrolment criteria, so those criteria are applied explicitly rather than assumed.

In [ ]:
eligible = labelled[labelled["trial_eligible"]]print(f"{len(labelled):,} sampled -> {len(eligible):,} trial-eligible "      f"({100 * len(eligible) / len(labelled):.0f}%)")display(eligible[["ejection_fraction", "edv_ml", "wall_thickness_cm",                  "peak_lvot_gradient_mmhg", "end_diastolic_pressure_mmhg",                  "e_over_e_prime"]].describe().round(2))

## The premise: baseline does not separate them

In [ ]:
safe = eligible[~eligible["over_responder"]]crash = eligible[eligible["over_responder"]]print(f"over-responders at or below the mid dose: {len(crash)} of {len(eligible)} "      f"({100 * len(crash) / len(eligible):.2f}%)")candidates = ["ejection_fraction", "wall_thickness_cm", "peak_lvot_gradient_mmhg",              "e_over_e_prime", "peak_strain_amplitude", "edv_ml"]figure, axes = plt.subplots(2, 3, figsize=(11, 5.6))for ax, name in zip(axes.ravel(), candidates):    bins = np.linspace(eligible[name].quantile(0.01), eligible[name].quantile(0.99), 24)    ax.hist(safe[name], bins=bins, density=True, color=style.SERIES[0], alpha=0.65,            label="tolerated", linewidth=0)    if len(crash):        ax.hist(crash[name], bins=bins, density=True, color=style.SERIES[1], alpha=0.75,                label="crossed the floor", linewidth=0)    ax.set_title(name.replace("_", " "), fontsize=9)    ax.set_yticks([])axes[0, 0].legend(fontsize=8)plt.tight_layout(); plt.show()

## What actually differsThe separation is in the *hidden* columns, which a clinic cannot see. This is the wholeargument in one table.

In [ ]:
hidden = [c for c in eligible.columns if c.startswith("true_")]if len(crash):    contrast = pd.DataFrame({        "tolerated": safe[hidden].median(),        "crossed the floor": crash[hidden].median(),    })    contrast["ratio"] = contrast["crossed the floor"] / contrast["tolerated"]    display(contrast.round(3).sort_values("ratio"))else:    print("no over-responders in this cohort; rerun with a larger n_base")